# 50-board transfer with Lazarus YOLO11s

Enable GPU and Internet. Attach the verified Mendeley PCB_Defect ZIP as a private input. This notebook downloads the published model, tests it before and after fixed-budget fine-tuning, and exports the result and checkpoint.


In [ ]:
from collections import Counter, defaultdict
from contextlib import ExitStack
import hashlib
import json
from pathlib import Path
import re
from zipfile import ZipFile

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working/pcb_50_board_transfer")
WORK.mkdir(parents=True, exist_ok=True)
ANN_NAME = "_annotations.coco.json"
ANN_SHA = "5d684e6d9f0a98de5015069228552d0f80ae4a539ddd288e60b7034e4f18f658"
IMAGE_SET_SHA = "60eee441e374315bd13b5f4907161381b59c4cd2de6ee8801948c27dc63d1959"
ZIP_SHA = "a5fb17e7009f96ffdb833cd9309b36da775545c8cc36a6dca3a8e9b4fbcb5f2a"
TRAIN_GROUPS = {"52", "55", "58", "62", "82"}
NAMES = ["short", "spur", "spurious_copper", "open", "mousebite",
         "hole_breakout", "conductor_scratch", "conductor_foreign_object",
         "base_material_foreign_object"]
COCO_TO_MODEL = {2: 4, 3: 3, 4: 0, 5: 1, 6: 2}
CLASSES = list(range(5))
DISPLAY_NAMES = dict(enumerate(["short", "spur", "spurious_copper", "open_circuit", "mouse_bite"]))

def digest(data):
    return hashlib.sha256(data).hexdigest()

def sha(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

def write_json(path, value):
    path.write_text(json.dumps(value, indent=2) + "\n")

def save_verified(path, data):
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f"Existing file differs: {path}")
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(data)

def prepare_dataset():
    extracted = next((p for p in INPUT.rglob(ANN_NAME) if sha(p) == ANN_SHA), None)
    with ExitStack() as stack:
        if extracted:
            annotations = extracted.read_bytes()
            image_dir = extracted.parent.parent / "images"
            read_image = lambda name: (image_dir / name).read_bytes()
        else:
            archive = next((p for p in INPUT.rglob("*.zip")
                            if p.stat().st_size == 158048862 and sha(p) == ZIP_SHA), None)
            if archive is None:
                raise FileNotFoundError("Attach the verified PCB_Defect ZIP as an input to this notebook.")
            zipped = stack.enter_context(ZipFile(archive))
            annotations = zipped.read("PCB_Defect/annotation/" + ANN_NAME)
            read_image = lambda name: zipped.read("PCB_Defect/images/" + name)

        assert digest(annotations) == ANN_SHA
        coco = json.loads(annotations)
        assert len(coco["images"]) == 230 and len(coco["annotations"]) == 1704
        assert {c["id"]: c["name"] for c in coco["categories"] if c["id"] > 0} == {
            1: "missing_pad", 2: "mouse_bite", 3: "open_circuit", 4: "short",
            5: "spur", 6: "spurious_copper"
        }
        by_image = defaultdict(list)
        for box in coco["annotations"]:
            if box["category_id"] in COCO_TO_MODEL:
                by_image[box["image_id"]].append(box)

        manifest = []
        image_set = hashlib.sha256()
        counts = defaultdict(Counter)
        seen = set()
        for row in sorted(coco["images"], key=lambda r: r["file_name"]):
            name = row["file_name"]
            assert re.fullmatch(r"pcb_defect_\d{3}\.jpg", name), name
            group = row["extra"]["name"].split("-", 1)[0]
            split = "train" if group in TRAIN_GROUPS else "holdout"
            data = read_image(name)
            image_sha = digest(data)
            assert image_sha not in seen, f"Duplicate image: {name}"
            seen.add(image_sha)
            image_set.update(name.encode() + b"\0" + bytes.fromhex(image_sha))
            labels = []
            width, height = row["width"], row["height"]
            for annotation in by_image[row["id"]]:
                cls = COCO_TO_MODEL[annotation["category_id"]]
                x, y, w, h = annotation["bbox"]
                assert 0 <= x < x+w <= width and 0 <= y < y+h <= height
                labels.append(f"{cls} {(x+w/2)/width:.10f} {(y+h/2)/height:.10f} {w/width:.10f} {h/height:.10f}")
                counts[split][DISPLAY_NAMES[cls]] += 1
            save_verified(WORK / "data/images" / split / name, data)
            label_path = WORK / "data/labels" / split / Path(name).with_suffix(".txt")
            label_data = "".join(line + "\n" for line in labels).encode()
            save_verified(label_path, label_data)
            manifest.append(dict(image=name, group=group, original_name=row["extra"]["name"],
                                 split=split, image_sha256=image_sha, label_sha256=digest(label_data),
                                 scored_boxes=len(labels)))
        assert image_set.hexdigest() == IMAGE_SET_SHA, "Image bytes differ from the audited dataset."
        assert Counter(row["split"] for row in manifest) == {"train": 50, "holdout": 180}
        for split in ("train", "holdout"):
            expected = {row["image"] for row in manifest if row["split"] == split}
            assert {p.name for p in (WORK / "data/images" / split).glob("*")} == expected
            assert {p.name for p in (WORK / "data/labels" / split).glob("*")} == {
                Path(name).with_suffix(".txt").name for name in expected
            }
        assert {s: set(c) for s, c in counts.items()} == {
            "train": set(DISPLAY_NAMES.values()), "holdout": set(DISPLAY_NAMES.values())
        }
        write_json(WORK / "split.json", dict(dataset_doi="10.17632/vdj74sngvn.1",
                   annotation_sha256=ANN_SHA, image_set_sha256=IMAGE_SET_SHA,
                   train_groups=sorted(TRAIN_GROUPS), group_rule="first prefix of COCO extra.name",
                   group_limit="Filename grouping and a local near-duplicate audit are proxies; layout independence is unknown.",
                   excluded_category="missing_pad", class_counts=counts, images=manifest))
        # JSON is valid YAML. The trainer can only validate against its 50 training boards.
        for name, validation in (("train_only", "train"), ("held_out", "holdout")):
            write_json(WORK / f"{name}.yaml", dict(path=str(WORK / "data"), train="images/train",
                       val=f"images/{validation}", names=NAMES))
        print("SPLIT READY | 50 training boards | 180 held-out boards")
        for split, values in counts.items():
            print(split, sum(values.values()), "boxes", dict(values))

prepare_dataset()


## Verify GPU and source checkpoint

The original YOLO class IDs are preserved for the five shared defects. Missing pads are outside the scored class set.


In [ ]:
import importlib
import importlib.metadata as metadata
import os
import shutil
import subprocess
import sys
from urllib.request import urlopen

os.environ["YOLO_CONFIG_DIR"] = str(WORK / "settings")
os.environ["MPLCONFIGDIR"] = str(WORK / "matplotlib")
os.environ["WANDB_MODE"] = "disabled"

try:
    installed = metadata.version("ultralytics")
except metadata.PackageNotFoundError:
    installed = None
if installed != "8.4.142":
    pins = WORK / "dependency_pins.txt"
    pins.write_text("".join(f"{name}=={metadata.version(name)}\n" for name in ("torch", "torchvision", "numpy")))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(pins),
                    "ultralytics==8.4.142"], check=True)

import torch
import torchvision
import ultralytics
from ultralytics import YOLO

assert ultralytics.__version__ == "8.4.142"
assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
DEVICE = 0
IMGSZ = 1024
boxes = torch.tensor([[0., 0., 10., 10.], [1., 1., 9., 9.]], device="cuda:0")
scores = torch.tensor([.9, .8], device="cuda:0")
assert torchvision.ops.nms(boxes, scores, .5).cpu().tolist() == [0]
del boxes, scores

SOURCE = WORK / "yolo11s-dspcbsd.pt"
WEIGHTS_URL = "https://github.com/Alpha-Ldz/lazarus/releases/download/weights-v1/yolo11s-dspcbsd.pt"
WEIGHTS_SHA = "834b8dc02142e866d1a98881c202e87b86a24a3b0048708024e6eb7cd94b3af5"
if not SOURCE.is_file():
    temporary = SOURCE.with_suffix(".download")
    with urlopen(WEIGHTS_URL, timeout=90) as response, temporary.open("wb") as stream:
        shutil.copyfileobj(response, stream)
    assert sha(temporary) == WEIGHTS_SHA
    temporary.replace(SOURCE)
assert sha(SOURCE) == WEIGHTS_SHA

def load_source():
    modules = {
        "torch.nn.modules.activation": "SiLU",
        "torch.nn.modules.batchnorm": "BatchNorm2d",
        "torch.nn.modules.container": "Sequential ModuleList",
        "torch.nn.modules.conv": "Conv2d",
        "torch.nn.modules.linear": "Identity",
        "torch.nn.modules.pooling": "MaxPool2d",
        "torch.nn.modules.upsampling": "Upsample",
        "ultralytics.nn.modules.block": "Attention Bottleneck C2PSA C3k C3k2 DFL PSABlock SPPF",
        "ultralytics.nn.modules.conv": "Concat Conv DWConv",
        "ultralytics.nn.modules.head": "Detect",
        "ultralytics.nn.tasks": "DetectionModel",
    }
    allowed = [getattr(importlib.import_module(m), n) for m, names in modules.items() for n in names.split()]
    previous = os.environ.get("TORCH_FORCE_WEIGHTS_ONLY_LOAD")
    os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = "1"
    try:
        with torch.serialization.safe_globals(allowed):
            model = YOLO(str(SOURCE))
    finally:
        if previous is None:
            os.environ.pop("TORCH_FORCE_WEIGHTS_ONLY_LOAD", None)
        else:
            os.environ["TORCH_FORCE_WEIGHTS_ONLY_LOAD"] = previous
    assert [model.names[i] for i in range(len(model.names))] == NAMES
    return model

print("ENVIRONMENT READY", torch.cuda.get_device_name(0), "Ultralytics", ultralytics.__version__)


## Compare on the same 180 boards

The 50 training boards and 180 evaluation boards have disjoint original filename prefixes. Layout independence is unknown. This fixed single-seed experiment uses the last epoch and keeps held-out labels out of training and checkpoint selection. Download the ZIP at the end.


In [ ]:
import csv
import zipfile
from IPython.display import FileLink, display

EPOCHS = 20
TRAIN_BATCH = 2
NMS_IOU = 0.4
EVAL_CONF = 0.001
training_dir = WORK / "train_50"
final_weights = training_dir / "weights/last.pt"

def evaluate(model, name, checkpoint):
    assert [model.names[i] for i in range(len(model.names))] == NAMES
    metrics = model.val(data=str(WORK / "held_out.yaml"), split="val", imgsz=IMGSZ,
        batch=1, device=DEVICE, workers=0, rect=False, conf=EVAL_CONF, iou=NMS_IOU,
        max_det=300, classes=CLASSES, agnostic_nms=False, augment=False,
        plots=False, save_json=False, project=str(WORK), name=name, exist_ok=True, verbose=False)
    assert set(map(int, metrics.box.ap_class_index)) == set(CLASSES)
    return dict(checkpoint_sha256=sha(checkpoint), mAP50=float(metrics.box.map50),
        mAP50_95=float(metrics.box.map), per_class_AP50={
            DISPLAY_NAMES[int(c)]: float(ap)
            for c, ap in zip(metrics.box.ap_class_index, metrics.box.ap50)
        })

baseline = evaluate(load_source(), "baseline_eval", SOURCE)
write_json(WORK / "baseline.json", baseline)
print(f"BASELINE | mAP50 {baseline['mAP50']:.2%} | mAP50–95 {baseline['mAP50_95']:.2%}")
torch.cuda.empty_cache()

# The recipe is fixed. The held-out 180 boards are absent from this training YAML.
recipe = dict(data=str(WORK / "train_only.yaml"), epochs=EPOCHS, imgsz=IMGSZ,
    batch=TRAIN_BATCH, nbs=8, device=DEVICE, workers=0, amp=False, cache=False,
    val=False, patience=0, save=True, plots=False, optimizer="AdamW", lr0=0.0001,
    lrf=0.1, cos_lr=True, warmup_epochs=1.0, seed=42, deterministic=True,
    rect=False, freeze=0, classes=CLASSES, augmentations=[], mosaic=0.0, mixup=0.0,
    copy_paste=0.0, cutmix=0.0, degrees=0.0, translate=0.03, scale=0.1, shear=0.0,
    perspective=0.0, fliplr=0.5, flipud=0.5, hsv_h=0.0, hsv_s=0.1, hsv_v=0.1,
    project=str(WORK), name=training_dir.name, exist_ok=False)
identity = dict(recipe=recipe, source_sha256=WEIGHTS_SHA, split_sha256=sha(WORK / "split.json"),
                ultralytics=ultralytics.__version__, torch=str(torch.__version__))
identity_file = WORK / "training_identity.json"

def completed_epochs():
    results = training_dir / "results.csv"
    if not results.is_file():
        return 0
    with results.open(newline="") as stream:
        return len(list(csv.DictReader(stream)))

if training_dir.exists():
    if (not final_weights.is_file() or completed_epochs() != EPOCHS
            or not identity_file.is_file() or json.loads(identity_file.read_text()) != identity):
        raise RuntimeError(f"Existing run is incomplete or differs. Preserve {training_dir}, then move it aside and rerun.")
else:
    write_json(identity_file, identity)
    load_source().train(**recipe)

assert final_weights.is_file() and completed_epochs() == EPOCHS
torch.cuda.empty_cache()
adapted = evaluate(YOLO(str(final_weights)), "adapted_eval", final_weights)

report = dict(train_boards=50, held_out_boards=180, scored_class_names=list(DISPLAY_NAMES.values()),
    imgsz=IMGSZ, nms_iou=NMS_IOU, ap_confidence_floor=EVAL_CONF,
    source_url=WEIGHTS_URL, checkpoint_selection="Last fixed epoch",
    split_sha256=sha(WORK / "split.json"),
    before=baseline, after=adapted,
    delta_mAP50=adapted["mAP50"]-baseline["mAP50"],
    delta_mAP50_95=adapted["mAP50_95"]-baseline["mAP50_95"])
write_json(WORK / "comparison.json", report)
print("\n=== 50-BOARD TRANSFER RESULTS ===")
for name, row in (("Before", baseline), ("After", adapted)):
    print(f"{name}: mAP50 {row['mAP50']:.2%} | mAP50–95 {row['mAP50_95']:.2%}")
    for label, ap in row["per_class_AP50"].items():
        print(f"  {label}: AP50 {ap:.2%}")
print(f"Change in mAP50: {100*report['delta_mAP50']:+.2f} percentage points")

backup = WORK / "pcb_50_board_results.zip"
temporary = backup.with_suffix(".tmp")
with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in (final_weights, training_dir / "results.csv", training_dir / "args.yaml",
                 WORK / "split.json", WORK / "training_identity.json", WORK / "comparison.json"):
        z.write(path, arcname=path.relative_to(WORK).as_posix())
temporary.replace(backup)
os.chdir(WORK.parent)
display(FileLink(str(backup.relative_to(WORK.parent))))
